# Séries temporais e exportação no GEE

Leia antes [Redutores espaciais, séries temporais e exportação](../aulas/05_redutores_series_e_exportacao.md).

Neste notebook, você vai:

1. Calcular estatísticas zonais (`reduceRegion`) sobre um polígono central no Parque Mãe Bonifácia em Cuiabá (MT), fixando `crs` e comparando diferentes níveis da pirâmide (`scale=10` versus `scale=100`).
2. Filtrar um único tile MGRS (`21LWC`) para evitar datas duplicadas em áreas de sobreposição de tiles.
3. Extrair uma série temporal anual de NDVI (2024) mantendo todas as datas (inclusive as encobertas por nuvens, com `pixels_validos`) na tabela e no CSV.
4. Plotar a série sem conectar lacunas da estação chuvosa com linhas enganosas.
5. Configurar tarefas assíncronas de exportação (`ee.batch.Export.image.toDrive` e `Export.table.toDrive`).

Dados referenciados: Contains modified Copernicus Sentinel data 2024.


In [ ]:
import os
import ee
import matplotlib.pyplot as plt
import pandas as pd

GEE_PROJECT_ID = os.environ.get("GEE_PROJECT_ID", "SEU_PROJECT_ID_AQUI")

# ee.Authenticate()
ee.Initialize(project=GEE_PROJECT_ID)


## 1. Preparar a coleção anual de NDVI (tile único `21LWC`)

Usamos um buffer circular de 100 m centrado no interior vegetado do Parque Mãe Bonifácia em Cuiabá (`[-56.11032, -15.57505]`, conferido na cena `S2B_21LWC_20240902_0_L2A`: 89% classe SCL vegetação, NDVI p10 = 0,46, mediana = 0,62, p90 = 0,70). Filtramos explicitamente `MGRS_TILE == "21LWC"` para garantir no máximo uma cena por data de passagem e não aplicamos um corte rígido de nuvens por cena inteira (`CLOUDY_PIXEL_PERCENTAGE`), deixando a máscara `SCL` e a contagem `pixels_validos` indicarem quais datas tiveram cobertura válida sobre o parque.


In [ ]:
CRS_NATIVO = "EPSG:32721"
poligono_estudo = ee.Geometry.Point([-56.11032, -15.57505]).buffer(100)


def preparar_s2_com_ndvi(imagem):
    scl = imagem.select("SCL")
    mascara = (
        scl.neq(1)
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )
    opticas = imagem.select(["B2", "B3", "B4", "B8"]).multiply(0.0001).updateMask(mascara)
    ndvi = opticas.normalizedDifference(["B8", "B4"]).rename("NDVI")
    return opticas.addBands(ndvi).copyProperties(imagem, ["system:time_start"])


colecao_2024 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(poligono_estudo)
    .filter(ee.Filter.eq("MGRS_TILE", "21LWC"))
    .filterDate("2024-01-01", "2025-01-01")
    .map(preparar_s2_com_ndvi)
)

print("Passagens no tile 21LWC em 2024:", colecao_2024.size().getInfo())


## 2. Comparar `reduceRegion` em escalas diferentes

Como `.median()` perde a projeção UTM nativa e cai para `EPSG:4326`, informamos `crs="EPSG:32721"` explicitamente. Observe como trocar `scale` de 10 m para 100 m lê outro nível da pirâmide de resolução (não é apenas uma reamostragem trivial) e altera contagem e desvio-padrão.


In [ ]:
mediana_seca = colecao_2024.filterDate("2024-07-01", "2024-10-01").median()

redutor = (
    ee.Reducer.mean()
    .combine(ee.Reducer.stdDev(), sharedInputs=True)
    .combine(ee.Reducer.count(), sharedInputs=True)
)

stats_10m = mediana_seca.select("NDVI").reduceRegion(
    reducer=redutor,
    geometry=poligono_estudo,
    crs=CRS_NATIVO,
    scale=10,
    maxPixels=1e7,
).getInfo()

stats_100m = mediana_seca.select("NDVI").reduceRegion(
    reducer=redutor,
    geometry=poligono_estudo,
    crs=CRS_NATIVO,
    scale=100,
    maxPixels=1e7,
).getInfo()

print("NDVI a 10 m (EPSG:32721):", stats_10m)
print("NDVI a 100 m (EPSG:32721):", stats_100m)


## 3. Extrair a série temporal completa no servidor e plotar localmente

Mantemos **todas as datas** em `serie_fc` (mesmo quando o parque estava coberto por nuvens e `pixels_validos` é 0), preservando o registro das lacunas na tabela e no CSV exportado. Só filtramos as observações com cobertura mínima no momento de desenhar o gráfico, e usamos marcadores sem linha contínua cruzando lacunas longas da estação chuvosa.


In [ ]:
def resumir_cena(imagem):
    stats = imagem.select("NDVI").reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True),
        geometry=poligono_estudo,
        crs=CRS_NATIVO,
        scale=10,
        maxPixels=1e7,
    )
    return ee.Feature(
        None,
        {
            "data": imagem.date().format("YYYY-MM-dd"),
            "ndvi_medio": stats.get("NDVI_mean"),
            "pixels_validos": stats.get("NDVI_count"),
        },
    )


serie_fc = colecao_2024.map(resumir_cena)

registros = [feicao["properties"] for feicao in serie_fc.getInfo()["features"]]
tabela_ndvi = pd.DataFrame(registros)
tabela_ndvi["data"] = pd.to_datetime(tabela_ndvi["data"])
tabela_ndvi = tabela_ndvi.sort_values("data").reset_index(drop=True)

total_datas = len(tabela_ndvi)
datas_sem_obs = int(tabela_ndvi["ndvi_medio"].isna().sum())
print(f"Total de passagens registradas: {total_datas} | Sem pixels válidos: {datas_sem_obs}")
print(tabela_ndvi.head(12))


In [ ]:
# Filtrar apenas para visualização, exigindo ao menos 70% de cobertura válida no polígono
min_pixels = tabela_ndvi["pixels_validos"].max() * 0.70
tabela_valida = tabela_ndvi[
    tabela_ndvi["ndvi_medio"].notna() & (tabela_ndvi["pixels_validos"] >= min_pixels)
]

fig, ax = plt.subplots(figsize=(9, 4))
# Marcadores sem linha contínua evitam interpolação visual sobre lacunas da estação chuvosa
ax.plot(tabela_valida["data"], tabela_valida["ndvi_medio"], marker="o", linestyle="none")
ax.set_title("Observações válidas de NDVI médio no Parque Mãe Bonifácia (Sentinel-2, 2024)")
ax.set_xlabel("Data da aquisição")
ax.set_ylabel("NDVI médio")
ax.set_ylim(-0.1, 1.0)
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


## 4. Exportação assíncrona para o Google Drive

As chamadas `.start()` abaixo estão comentadas para não disparar gravações automáticas no seu Google Drive. Usamos `EPSG:32721` para preservar a grade UTM nativa do Sentinel-2 sem reamostragem extra (para padrão cartográfico brasileiro SIRGAS 2000, use `EPSG:31981`).


In [ ]:
tarefa_csv = ee.batch.Export.table.toDrive(
    collection=serie_fc,
    description="serie_ndvi_2024_csv",
    folder="gee_estudos",
    fileNamePrefix="serie_ndvi_2024",
    fileFormat="CSV",
)

tarefa_geotiff = ee.batch.Export.image.toDrive(
    image=mediana_seca.select(["B4", "B3", "B2", "B8", "NDVI"]).toFloat(),
    description="s2_mediana_seca_2024_tif",
    folder="gee_estudos",
    fileNamePrefix="s2_mediana_seca_2024",
    region=poligono_estudo.bounds(),
    scale=10,
    crs=CRS_NATIVO,
    maxPixels=1e8,
)

# Descomente para iniciar no servidor do Earth Engine:
# tarefa_csv.start()
# tarefa_geotiff.start()
# print("CSV:", tarefa_csv.status())
# print("GeoTIFF:", tarefa_geotiff.status())
print("Tarefas configuradas (ainda não iniciadas).")
